Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers
- pip intall tiktoken

## fsdf
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [1]:
import os
import hashlib
from langchain_community.document_loaders import PyPDFLoader


In [2]:
import os
def archivo_contenido(archivo):
    if not os.path.exists(archivo): # si no existe el archivo lo crea
        with open(archivo, 'w') as file:
            pass

    # verifica su contenido
    with open(archivo, 'r') as file:
        docs_procesados= list(file.read().splitlines())
    
    # print(f'Documentos procesados: {docs_procesados}')
    return docs_procesados

In [5]:
procesados = archivo_contenido('procesados.txt')

In [6]:
ruta_docs_pdf= '../doc_pdf'
# carpeta_embeddings = ''

def generate_no_procesados(ruta_docs_pdf):
    procesados = archivo_contenido('procesados.txt')
    docs_no_procesados= []
    # verificamos los archivos en carpeta de docs
    for filename in os.listdir(ruta_docs_pdf):
        # verificar si el archivo se encuentra en procesados.txt
        if filename not in procesados:
            # print('El archivo no ha sido procesado')
            ruta_completa= os.path.join(ruta_docs_pdf,filename)
            docs_no_procesados.append(ruta_completa)
        # else:
        #     print('Todos los archivos han sido procesados')

    print(f'Documentos para procesar:\n  {docs_no_procesados}')
    return docs_no_procesados

docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  []


## Empieza el procesamiento

Extracción del texto 

In [7]:
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    print('✅ Extracción realizada')
    return pages


Limpieza del texto

In [8]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    return text.strip()


Creacción de la metadata, estructura planteada:
- documento_id
- nombre documento
- numero pagina
- total_pages

In [9]:
from pathlib import Path
import hashlib

def generate_metadata(ruta_completa, pages):
    filename = Path(ruta_completa).name
    document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo
    total_pages = pages[0].metadata['total_pages']
    docs_metadata = []
    for page in pages:
        page_number = page.metadata['page_label'] # númeración correcta de la página
        
        metadata = {
            "document_id": document_id,
            "filename": filename,
            "page_number": page_number,
            "total_pages": total_pages,
        }
        page.page_content = clean_text(page.page_content) # cleaned_text = clean_text(page.page_content)
        
        docs_metadata.append(
            {
                'text': page.page_content, #cleaned_text, 
                'metadata': metadata
            }
        )
    print('✅ Generación Documentos con Metadata (Limpieza por página)')
    return docs_metadata

Generación de embeddings

In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [12]:
import tiktoken # estimar la cantidad de token
import time

def costo_tokens(tokens):
    costo = tokens*0.02 /10**6 # 1 millon de tokens equivale a 0.02 dólares

    return f'   Tokens: {tokens}\n   Costo Tokens: ${costo:.4f}'


def generate_embedd(docs_metadata):
    print(f'   Generando embedding: ...')
    modelo_openai = "text-embedding-3-small"
    encoding= tiktoken.encoding_for_model(modelo_openai)

    docs_embedd = []
    total_tokens= 0


    for doc in docs_metadata:
        texto= doc['text']

        # Generación de número de tokens
        tokens= encoding.encode(texto)
        nro_tokens = len(tokens)
        total_tokens += nro_tokens

        doc['metadata']['token']=nro_tokens # añado los tokens a la metadata
    
        start= time.time()
        #  Generación de embeddings
        response = cliente.embeddings.create(
            input= texto, 
            model = modelo_openai
        )
        finish= time.time()
        embedding= response.data[0].embedding
        
        # embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
        docs_embedd.append({
            'vector': embedding,  #con openai, directamente el embedding
            'text': texto, 
            'metadata': doc['metadata'] 

        })
    print(costo_tokens(total_tokens))
    print(f'   Tiempo del embedding: {finish-start:.4f} segundos')
    print('✅ Generación de Embeddings')
    return docs_embedd

# comprobar con lo que sale en playground

Exportación embedding

In [48]:
ruta_completa

'../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf'

In [34]:
# filename = Path(ruta_completa).name

import json
def exportacion_json(docs_embeding,filename):
    with open(f"../json_embedding/{filename}.json", "w", encoding="utf-8") as file:
        json.dump(docs_embeding, file, ensure_ascii=False, indent=2)

    # luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
    with open('procesados.txt', 'a', encoding='utf-8') as file:
        file.write(filename+"\n")

    print('✅ Exportacción realizada')

# exportacion_json(filename)

## Funcion completa
**def procesamiento():**
- extracion texto
- limpieza por pagina
- creacion de docs_metadata
- obtencion de embedding
- exportación de embedding

In [18]:
docs_no_procesados

[]

In [16]:
if docs_no_procesados: # 
    print('hay doc por procesarr')
else:
    print('Todo ready')

Todo ready


In [37]:
import tqdm
prueba = ['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

# for ruta_archivo in  tqdm.tqdm(prueba):#docs_no_procesados:
def proceso_completo(docs_no_procesados):
    '''
    Parámetro de entrada: Lista con todas las rutas de los archivos no procesados
    '''
    if docs_no_procesados:
        for ruta_archivo in  docs_no_procesados:#prueba:
            filename = Path(ruta_archivo).name
            print(f'📌 Generando Embeddings para {filename} ...') # CAMBIO AQUI
            # definir una funcion para aplicar  
            time1= time.time()
            pags=extraccion_page(ruta_archivo)
            docs_metadata = generate_metadata(ruta_archivo, pags)
            docs_embedd= generate_embedd(docs_metadata)
            exportacion_json(docs_embedd,filename)
            time3=time.time()
            segundos= time3-time1 
            print(f'\nTiempo total: {segundos:.2f} segundos - {segundos/60:.2f} minutos')
            print('🎉 Realizado: Embeddings Generados Correctamente.\n\n')

    else:
        print('No hay documentos por procesar')
    

In [55]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 223221647-ECN-BusinessPath-fulldoc.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 1938
   Costo Tokens: $0.0000
Tiempo del embedding: 0.5894 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 3.92 segundos
🎉 Realizado: Embeddings Generados Correctamente.



Ya ahora que tengo el embedding demo vamos a modularizar

In [70]:
# actualizamos para docs_no_procesados
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


In [71]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 190165
   Costo Tokens: $0.0038
   Tiempo del embedding: 0.2444 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 184.87 segundos
🎉 Realizado: Embeddings Generados Correctamente.




In [19]:
proceso_completo(docs_no_procesados)

No hay documentos por procesar


In [20]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/605838498-EMyth-Annual-Plan-2023.pdf']


In [21]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 605838498-EMyth-Annual-Plan-2023.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 4712
   Costo Tokens: $0.0001
   Tiempo del embedding: 0.1757 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 8.74 segundos - 0.15 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [25]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/383663964-founder-to-ceo-how-to-build-a-great-company-matt-mochary.pdf', '../doc_pdf/465076208-The-Great-CEO-Within.pdf']


In [26]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 383663964-founder-to-ceo-how-to-build-a-great-company-matt-mochary.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 45514
   Costo Tokens: $0.0009
   Tiempo del embedding: 0.1551 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 29.82 segundos - 0.50 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 465076208-The-Great-CEO-Within.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 61821
   Costo Tokens: $0.0012
   Tiempo del embedding: 0.1669 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 64.03 segundos - 1.07 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [31]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/El juego infinito (Gestión del conocimiento) (Spanish Edition).pdf', '../doc_pdf/Marshall-Ganz-People-Power-and-Change.pdf', '../doc_pdf/354381363-Holocracia.pdf']


In [33]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para El juego infinito (Gestión del conocimiento) (Spanish Edition).pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 115394
   Costo Tokens: $0.0023
   Tiempo del embedding: 3.3500 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 77.76 segundos - 1.30 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para Marshall-Ganz-People-Power-and-Change.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 12304
   Costo Tokens: $0.0002
   Tiempo del embedding: 0.2302 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 8.02 segundos - 0.13 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 354381363-Holocracia.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Gen

In [39]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/638352159-QUIEN-NO-COMO-Dan-Sullivan.pdf', '../doc_pdf/428339485-Quarterly-Plan-Guide.pdf']


In [40]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 638352159-QUIEN-NO-COMO-Dan-Sullivan.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 75535
   Costo Tokens: $0.0015
   Tiempo del embedding: 0.2153 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 68.96 segundos - 1.15 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 428339485-Quarterly-Plan-Guide.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 2069
   Costo Tokens: $0.0000
   Tiempo del embedding: 0.1946 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 2.62 segundos - 0.04 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [42]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/The Customer Service Revolution PDF.pdf', '../doc_pdf/593900682-EL-ALMANAKE-DE-NAVAL-RAVIKANT.pdf', '../doc_pdf/544611339-Profit-First-a-Simple-System-to-Transform-Any-Business-From-a-Cash-eating-Monster-to-a-Money-making-Machine-PDFDrive.pdf', '../doc_pdf/The Customer-Funded Business PDF.pdf', '../doc_pdf/767870319-Hyper-Sales-Growth-Jack-Daly.pdf']


In [43]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para The Customer Service Revolution PDF.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 19058
   Costo Tokens: $0.0004
   Tiempo del embedding: 0.2501 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 39.61 segundos - 0.66 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 593900682-EL-ALMANAKE-DE-NAVAL-RAVIKANT.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 64615
   Costo Tokens: $0.0013
   Tiempo del embedding: 0.1511 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 42.76 segundos - 0.71 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 544611339-Profit-First-a-Simple-System-to-Transform-Any-Business-From-a-Cash-eating-Monster-to-a-Money-making-Machine-PDFDrive.pdf ...
✅ Extra

In [3]:
import langchain
import langchain_community